<a href="https://colab.research.google.com/github/ashitasingh1230-commits/netflix_content_analysis/blob/main/01_netflix_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Load and inspect the data**

In [1]:
import pandas as pd
df=pd.read_csv('netflix_titles.csv')
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


# **Check the full size and missing data**

In [2]:
print(df.shape)
df.info()

(8807, 12)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


# **Check the type breakdown (Movie vs TV Show)**

In [3]:
df['type'].value_counts()

,count
type,
Movie,6131
TV Show,2676


# **Look at the genre column more closely**

In [4]:
df['listed_in'].head(20)

,listed_in
0,Documentaries
1,"International TV Shows, TV Dramas, TV Mysteries"
2,"Crime TV Shows, International TV Shows, TV Act..."
3,"Docuseries, Reality TV"
4,"International TV Shows, Romantic TV Shows, TV ..."
5,"TV Dramas, TV Horror, TV Mysteries"
6,Children & Family Movies
7,"Dramas, Independent Movies, International Movies"
8,"British TV Shows, Reality TV"
9,"Comedies, Dramas"


# **Split the genre text into a list**

In [5]:
df['genre_list']=df['listed_in'].str.split( ', ')
df[['title','listed_in','genre_list']].head()

,title,listed_in,genre_list
0,Dick Johnson Is Dead,Documentaries,[Documentaries]
1,Blood & Water,"International TV Shows, TV Dramas, TV Mysteries","[International TV Shows, TV Dramas, TV Mysteries]"
2,Ganglands,"Crime TV Shows, International TV Shows, TV Act...","[Crime TV Shows, International TV Shows, TV Ac..."
3,Jailbirds New Orleans,"Docuseries, Reality TV","[Docuseries, Reality TV]"
4,Kota Factory,"International TV Shows, Romantic TV Shows, TV ...","[International TV Shows, Romantic TV Shows, TV..."


In [6]:
type(df['genre_list'].iloc[0])

list

# **Build the genre junction table**

In [7]:
genres_df=df[['show_id','genre_list']].explode('genre_list')
genres_df.columns=['show_id','genre']
genres_df.head(10)

,show_id,genre
0,s1,Documentaries
1,s2,International TV Shows
1,s2,TV Dramas
1,s2,TV Mysteries
2,s3,Crime TV Shows
2,s3,International TV Shows
2,s3,TV Action & Adventure
3,s4,Docuseries
3,s4,Reality TV
4,s5,International TV Shows


# **Check the new table's size**

In [8]:
genres_df.shape

(19323, 2)

# **Prepare the main titles table (without the messy genre columns)**

In [9]:
titles_df=df.drop(columns= ['listed_in','genre_list'])
titles_df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,In a city of coaching centers known to train I...


# **Create the database with both tables**

In [10]:
import sqlite3
conn=sqlite3.connect('netflix.db')
titles_df.to_sql('titles',conn,if_exists='replace',index= False)
genres_df.to_sql('genres',conn,if_exists='replace',index=False)

19323

# **join both tables**

In [11]:
query= """
select t.title ,t.type , g.genre
from titles t
join genres g on t.show_id = g.show_id
limit 10
"""
pd.read_sql(query,conn)

,title,type,genre
0,Dick Johnson Is Dead,Movie,Documentaries
1,Blood & Water,TV Show,International TV Shows
2,Blood & Water,TV Show,TV Dramas
3,Blood & Water,TV Show,TV Mysteries
4,Ganglands,TV Show,Crime TV Shows
5,Ganglands,TV Show,International TV Shows
6,Ganglands,TV Show,TV Action & Adventure
7,Jailbirds New Orleans,TV Show,Docuseries
8,Jailbirds New Orleans,TV Show,Reality TV
9,Kota Factory,TV Show,International TV Shows


# **Most common genres overall**

In [12]:
query= """
select g.genre , count(*) as title_count
from genres g
group by g.genre
order by title_count desc
limit 10
"""
pd.read_sql(query,conn)

,genre,title_count
0,International Movies,2752
1,Dramas,2427
2,Comedies,1674
3,International TV Shows,1351
4,Documentaries,869
5,Action & Adventure,859
6,TV Dramas,763
7,Independent Movies,756
8,Children & Family Movies,641
9,Romantic Movies,616


# **Genre counts by country**

In [13]:
query= """
select t.country , g.genre, count(*) as total_count
from titles t
join genres g on t.show_id=g.show_id
where t.country is not null
group by g.genre , t.country
order by total_count desc
limit 20
"""
pd.read_sql(query,conn)

,country,genre,total_count
0,India,International Movies,817
1,India,Dramas,620
2,United States,Dramas,591
3,United States,Comedies,524
4,United States,Documentaries,411
5,India,Comedies,308
6,United States,Independent Movies,302
7,United States,Children & Family Movies,272
8,United States,Action & Adventure,249
9,United States,TV Comedies,234


# **Fix mix data**

In [16]:
query = """
select t.country , g.genre , count(*) as total_count , rank() over (partition by t.country order by count(*) desc) as genre_rank
from titles t
join genres g on t.show_id=g.show_id
where t.country is not null
group by t.country , g.genre
"""
result=pd.read_sql(query,conn)
result.head(20)



,country,genre,total_count,genre_rank
0,", France, Algeria",International Movies,1,1
1,", France, Algeria",Independent Movies,1,1
2,", France, Algeria",Dramas,1,1
3,", South Korea",TV Dramas,1,1
4,", South Korea",International TV Shows,1,1
5,Argentina,International Movies,29,1
6,Argentina,Spanish-Language TV Shows,16,2
7,Argentina,International TV Shows,15,3
8,Argentina,Dramas,14,4
9,Argentina,Stand-Up Comedy,8,5


# **Filter to just the top genre per country**

In [17]:
query = """
with ranked_genres as (
select t.country , g.genre , count(*) as total_count , rank() over(partition by t.country order by count(*) desc) as genre_rank
from titles t
join genres g on t.show_id=g.show_id
where t.country is not null
group by t.country , g.genre
)
select country , genre , total_count
from ranked_genres
where genre_rank = 1
order by total_count desc
limit 15
"""
pd.read_sql(query,conn)

,country,genre,total_count
0,India,International Movies,817
1,United States,Dramas,591
2,United Kingdom,British TV Shows,196
3,South Korea,International TV Shows,149
4,Japan,International TV Shows,141
5,Spain,International Movies,90
6,Egypt,International Movies,89
7,Nigeria,International Movies,81
8,Turkey,International Movies,74
9,Indonesia,International Movies,73


Excluding the U.S. and U.K., nearly every country's top-ranked genre was 'International Movies' or 'International TV Shows.' Rather than representing an actual content preference, this most likely reflects Netflix's own catalog labeling convention, which tags non-U.S. content as 'International' by default. This suggests genre-level analysis should be interpreted carefully — the label reveals more about the platform's categorization logic than about audience taste in those countries.

# **Content added by year and type**

In [20]:

query = """
select substr(date_added,-4,4) as year_added , type,count(*) as total_count
from titles
where date_added is not null
group by type ,year_added
order by year_added

"""
pd.read_sql(query,conn)


,year_added,type,total_count
0,2008,Movie,1
1,2008,TV Show,1
2,2009,Movie,2
3,2010,Movie,1
4,2011,Movie,13
5,2012,Movie,3
6,2013,Movie,6
7,2013,TV Show,5
8,2014,Movie,19
9,2014,TV Show,5
